# Bitcoin On-Chain Data — Schema Exploration

In [4]:
import os
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    SET s3_access_key_id='{os.environ["AWS_ACCESS_KEY_ID"]}';
    SET s3_secret_access_key='{os.environ["AWS_SECRET_ACCESS_KEY"]}';
    SET s3_region='ap-southeast-1';
""")

S3_PATH = "s3://input-btc-bq/bitcoin/transactions/**/*.parquet"
print("Connected to DuckDB and S3")

Connected to DuckDB and S3


In [5]:
# Schema — column names and types
#con.sql(f"DESCRIBE SELECT * FROM read_parquet('{S3_PATH}')").show()
df = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{S3_PATH}')").df()
df

,column_name,column_type,null,key,default,extra
0,hash,VARCHAR,YES,None,None,None
1,size,BIGINT,YES,None,None,None
2,virtual_size,BIGINT,YES,None,None,None
3,version,BIGINT,YES,None,None,None
4,lock_time,BIGINT,YES,None,None,None
5,block_hash,VARCHAR,YES,None,None,None
6,block_number,BIGINT,YES,None,None,None
7,block_timestamp,TIMESTAMP WITH TIME ZONE,YES,None,None,None
8,block_timestamp_month,DATE,YES,None,None,None
9,input_count,BIGINT,YES,None,None,None


In [17]:
# Sample rows

df = con.sql(f"""
    SELECT * 
    FROM read_parquet('{S3_PATH}', union_by_name=true) 
    LIMIT 5
""").df()
df

,hash,size,virtual_size,version,lock_time,block_hash,block_number,block_timestamp,block_timestamp_month,input_count,output_count,input_value,output_value,is_coinbase,fee,inputs,outputs,date
0,4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc7...,204,204,1,0,000000000019d6689c085ae165831e934ff763ae46a2a6...,0,2009-01-03 18:15:05+00:00,2009-01-01,0,1,NaN,5.000000e+09,True,0.0,[],[{'addresses': ['1A1zP1eP5QGefi2DMPTfTL5SLmv7D...,2009-01-03
1,999e1c837c76a1b7fbb7e57baf87b309960f5ffefbf2a9...,134,134,1,0,0000000082b5015589a3fdf2d4baff403e6f0be035a5d9...,3,2009-01-09 03:02:53+00:00,2009-01-01,0,1,NaN,5.000000e+09,True,0.0,[],[{'addresses': ['1FvzCLoTPGANNjWoUo6jUGuAG3wg1...,2009-01-09
2,f8325d8f7fa5d658ea143629288d0530d2710dc9193ddc...,134,134,1,0,0000000097be56d606cdd9c54b04d4747e957d3608abe6...,11,2009-01-09 04:12:40+00:00,2009-01-01,0,1,NaN,5.000000e+09,True,0.0,[],[{'addresses': ['1dyoBoF5vDmPCxwSsUZbbYhA5qjAf...,2009-01-09
3,e1afd89295b68bc5247fe0ca2885dd4b8818d7ce430faa...,134,134,1,0,0000000080f17a0c5a67f663a9bc9969eb37e81666d932...,14,2009-01-09 04:33:09+00:00,2009-01-01,0,1,NaN,5.000000e+09,True,0.0,[],[{'addresses': ['1DMGtVnRrgZaji7C9noZS3a1QtoaA...,2009-01-09
4,df2b060fa2e5e9c8ed5eaf6a45c13753ec8c63282b2688...,134,134,1,0,000000004ebadb55ee9096c9a2f8880e09da59c0d68b1c...,4,2009-01-09 03:16:28+00:00,2009-01-01,0,1,NaN,5.000000e+09,True,0.0,[],[{'addresses': ['15ubicBBWFnvoZLT7GiU2qxjRaKJP...,2009-01-09


In [16]:
# Row count and date range
df = con.sql(f"""
    SELECT
        COUNT(*)              AS total_transactions,
        MIN(block_timestamp)  AS earliest,
        MAX(block_timestamp)  AS latest
    FROM read_parquet('{S3_PATH}', union_by_name=true)
""").df()
df

,total_transactions,earliest,latest
0,9480,2009-01-03 18:15:05+00:00,2009-03-31 23:42:42+00:00


In [15]:
df = con.sql(f"""
    SELECT
        DATE(block_timestamp) AS date,
        COUNT(*)              AS tx_count
    FROM read_parquet('{S3_PATH}', union_by_name=true)
    GROUP BY 1
    ORDER BY 1
""").df()
df

,date,tx_count
0,2009-01-03,1
1,2009-01-09,14
2,2009-01-10,61
3,2009-01-11,93
4,2009-01-12,101
...,...,...
78,2009-03-27,106
79,2009-03-28,122
80,2009-03-29,116
81,2009-03-30,120


In [14]:
df = con.sql(f"""
    SELECT
        t.hash,
        t.block_timestamp,
        t.block_number,
        t.fee,
        t.is_coinbase,
        inp."index"                AS input_index,
        inp.addresses[1]           AS sender_address,
        inp.value                  AS input_value_satoshi,
        inp.spent_transaction_hash AS spent_tx_hash,
        inp.type                   AS input_type
    FROM read_parquet('{S3_PATH}', union_by_name=true) t,
    UNNEST(t.inputs) AS u(inp)
""").df()
df

,hash,block_timestamp,block_number,fee,is_coinbase,input_index,sender_address,input_value_satoshi,spent_tx_hash,input_type
0,a16f3ce4dd5deb92d98ef5cf8afeaf0775ebca408f708b...,2009-01-12 06:02:13+00:00,181,0.0,False,0,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,4.000000e+09,f4184fc596403b9d638783cf57adfe4c75c605f6356fbc...,pubkey
1,591e91f809d716912ca1d4a9295e70c3e78bab077683f7...,2009-01-12 06:12:16+00:00,182,0.0,False,0,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,3.000000e+09,a16f3ce4dd5deb92d98ef5cf8afeaf0775ebca408f708b...,pubkey
2,12b5633bad1f9c167d523ad1aa1947b2732a865bf5414e...,2009-01-12 06:34:22+00:00,183,0.0,False,0,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,2.900000e+09,591e91f809d716912ca1d4a9295e70c3e78bab077683f7...,pubkey
3,828ef3b079f9c23829c56fe86e85b4a69d9e06e5b54ea5...,2009-01-12 20:04:20+00:00,248,0.0,False,0,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,2.800000e+09,12b5633bad1f9c167d523ad1aa1947b2732a865bf5414e...,pubkey
4,f4184fc596403b9d638783cf57adfe4c75c605f6356fbc...,2009-01-12 03:30:25+00:00,170,0.0,False,0,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,5.000000e+09,0437cd7f8525ceed2324359c2d0ba26006d92d856a9c20...,pubkey
...,...,...,...,...,...,...,...,...,...,...
603,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,27,1GsA8fMqX7AcXuXjvmyMMDBHEWXHwD5yfQ,5.000000e+09,dc314a59df2ecc11b2776121f55cd57a9bba6dd6609f28...,pubkey
604,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,28,1Apogsw3ne1uyccG5EGiraTL6y12KTYQcn,5.000000e+09,bb4b7ce80dd99de109d4acc94c4f85e2dc274b42d73f85...,pubkey
605,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,29,1F9C6XvdF5yWCnizqxNa6h96r3npJTLn5k,5.000000e+09,43dd64b3a6daf90fa6b2a4cc45c8b5448e0123de048498...,pubkey
606,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,30,1G2qyVU6dYm4i4YHBKj2PjwhhpPpuhcNU6,5.000000e+09,33556a0fa0b5a44313c1cf5c3381baa912bae37d379948...,pubkey


In [13]:
# Explode outputs — one row per output
df = con.sql(f"""
    SELECT
        t.hash,
        t.block_timestamp,
        t.block_number,
        out."index"        AS output_index,
        out.addresses[1]   AS receiver_address,
        out.value           AS output_value_satoshi,
        out.type            AS output_type
    FROM read_parquet('{S3_PATH}', union_by_name=true) t,
    UNNEST(t.outputs) AS u(out)
""").df()
df

,hash,block_timestamp,block_number,output_index,receiver_address,output_value_satoshi,output_type
0,4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc7...,2009-01-03 18:15:05+00:00,0,0,1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa,5.000000e+09,pubkey
1,999e1c837c76a1b7fbb7e57baf87b309960f5ffefbf2a9...,2009-01-09 03:02:53+00:00,3,0,1FvzCLoTPGANNjWoUo6jUGuAG3wg1w4YjR,5.000000e+09,pubkey
2,f8325d8f7fa5d658ea143629288d0530d2710dc9193ddc...,2009-01-09 04:12:40+00:00,11,0,1dyoBoF5vDmPCxwSsUZbbYhA5qjAfBTx9,5.000000e+09,pubkey
3,e1afd89295b68bc5247fe0ca2885dd4b8818d7ce430faa...,2009-01-09 04:33:09+00:00,14,0,1DMGtVnRrgZaji7C9noZS3a1QtoaAN2uRG,5.000000e+09,pubkey
4,df2b060fa2e5e9c8ed5eaf6a45c13753ec8c63282b2688...,2009-01-09 03:16:28+00:00,4,0,15ubicBBWFnvoZLT7GiU2qxjRaKJPdkDMG,5.000000e+09,pubkey
...,...,...,...,...,...,...,...
9502,a6c83e3e2721e303647b5de8f2d3bf90ff9427f5a36fd3...,2009-03-31 00:19:18+00:00,9281,0,1FFK7jqKLALqZtSVDRWDkqdHJw2hzvaYf2,5.000000e+09,pubkey
9503,fd50e29fbf4ef37c0f655354524b0e8dd5ce341f09d9eb...,2009-03-31 19:27:09+00:00,9373,0,1Mk3FaBJstFv5qthCnkMp8FA5dquqY868N,5.000000e+09,pubkey
9504,820347cba30ce4c464dfe923c86b8a9dcc4a52630fc298...,2009-03-31 05:58:36+00:00,9305,0,1JsqaUuvjvqj95Gc6xJCNCXQ4f27kd4Jdb,5.000000e+09,pubkey
9505,eb2d5f372ad48def90a7518429182c16da6ffb179e7225...,2009-03-31 16:59:06+00:00,9361,0,1GkB52aM4NCuQg2gDQeh5qKVChyrgEixu8,5.000000e+09,pubkey


In [12]:
df = con.sql(f"""
    SELECT
        t.hash,
        t.block_timestamp,
        t.block_number,
        t.fee / 1e8 AS fee_btc,
        t.is_coinbase,
        inp.addresses[1] AS from_address,
        out.addresses[1] AS to_address,
        out.value AS btc_value
    FROM read_parquet('{S3_PATH}', union_by_name=true) t,
    UNNEST(t.inputs) AS u(inp),
    UNNEST(t.outputs) AS v(out)
""").df()
df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,hash,block_timestamp,block_number,fee_btc,is_coinbase,from_address,to_address,btc_value
0,a16f3ce4dd5deb92d98ef5cf8afeaf0775ebca408f708b...,2009-01-12 06:02:13+00:00,181,0.0,False,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,1DUDsfc23Dv9sPMEk5RsrtfzCw5ofi5sVW,1.000000e+09
1,a16f3ce4dd5deb92d98ef5cf8afeaf0775ebca408f708b...,2009-01-12 06:02:13+00:00,181,0.0,False,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,3.000000e+09
2,591e91f809d716912ca1d4a9295e70c3e78bab077683f7...,2009-01-12 06:12:16+00:00,182,0.0,False,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,1LzBzVqEeuQyjD2mRWHes3dgWrT9titxvq,1.000000e+08
3,591e91f809d716912ca1d4a9295e70c3e78bab077683f7...,2009-01-12 06:12:16+00:00,182,0.0,False,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,2.900000e+09
4,12b5633bad1f9c167d523ad1aa1947b2732a865bf5414e...,2009-01-12 06:34:22+00:00,183,0.0,False,12cbQLTFMXRnSzktFkuoG3eHoMeFtpTu3S,13HtsYzne8xVPdGDnmJX8gHgBZerAfJGEf,1.000000e+08
...,...,...,...,...,...,...,...,...
635,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,1GsA8fMqX7AcXuXjvmyMMDBHEWXHwD5yfQ,12higDjoCCNXSA95xZMWUdPvXNmkAduhWv,1.600000e+11
636,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,1Apogsw3ne1uyccG5EGiraTL6y12KTYQcn,12higDjoCCNXSA95xZMWUdPvXNmkAduhWv,1.600000e+11
637,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,1F9C6XvdF5yWCnizqxNa6h96r3npJTLn5k,12higDjoCCNXSA95xZMWUdPvXNmkAduhWv,1.600000e+11
638,85b6f48c8e10d8e1df4c5e3b64f6209d6bd8a3ad0af7e3...,2009-03-31 15:46:29+00:00,9354,0.0,False,1G2qyVU6dYm4i4YHBKj2PjwhhpPpuhcNU6,12higDjoCCNXSA95xZMWUdPvXNmkAduhWv,1.600000e+11
